# CORNEAL-TRUST: Full Cloud Training (Kaggle / Colab T4)

Trains the Phase 2 segmentation U-Net and the Phase 3 severity classifier
at full resolution (384x384) on a free T4 GPU.

### Setup before running
1. Upload this repo + the 4 datasets to Google Drive (or a Kaggle dataset):
   - `CORNEAL_TRUST/` (repo)
   - `CORN-1/`, `CORN-2/`, `CORN-3/`, `CORN1500/` (datasets)
2. Update `DATA_PARENT` below to the folder that contains the datasets.
3. Run cells in order.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, sys, shutil
# --- EDIT THIS ---
DATA_PARENT = '/content/drive/MyDrive/corneal'  # folder holding CORN-1/.../CORN1500
REPO = '/content/CORNEAL_TRUST'
# -----------------

os.chdir('/content')

# Bring the repo in (already cloned: git clone https://github.com/gsahoo211004/CORNEAL-TRUST.git)
if not os.path.exists(REPO):
    !git clone https://github.com/gsahoo211004/CORNEAL-TRUST.git
sys.path.insert(0, REPO)

# Symlink the datasets as siblings of the repo (scripts resolve ../CORN-1 etc.)
for name in ['CORN-1', 'CORN-2', 'CORN-3', 'CORN1500']:
    src = os.path.join(DATA_PARENT, name)
    dst = os.path.join('/content', name)
    if os.path.exists(src) and not os.path.exists(dst):
        os.symlink(src, dst)
        print('linked', name)
    else:
        print('missing or already present:', name)

In [ ]:
%pip install -q -r "$REPO/requirements.txt" 2>&1 | tail -5
# If torch CPU was installed by default, reinstall the CUDA build:
%pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121 2>&1 | tail -3

In [ ]:
%cd "$REPO"
import torch
print('cuda available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

## 1. Segmentation U-Net (Phase 2) -- CORN-1
Full 50-epoch run, early stopping on validation Dice.

In [ ]:
!python scripts/train_segmentation.py 

import glob
ckpt = glob.glob('outputs/checkpoints/unet_corn1.pt')
print(ckpt)

## 2. Severity classifier (Phase 3) -- CORN1500 + CORN-3 val
Full 50-epoch run, early stopping on validation accuracy.

In [ ]:
!python scripts/train_severity.py

import glob
ckpt = glob.glob('outputs/checkpoints/severity_corn1500.pt')
print(ckpt)

In [ ]:
# Mirror checkpoints + logs back to Drive for safekeeping
out = os.path.join(DATA_PARENT, 'CORNEAL_TRUST_outputs')
os.makedirs(out, exist_ok=True)
shutil.copytree('outputs', os.path.join(out, 'outputs'), dirs_exist_ok=True)
print('saved to', out)